# 🎯 [ISR02_M04] 생성형 AI 조작 이미지/텍스트 탐지 및 출처 계정 신뢰도 평가 모델 고도화
### Google Colab 실습 환경 (정보·감시·정찰(ISR) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/ISR02/isr02_m04_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 정보·감시·정찰(ISR) (Level 2) |
| **미션 ID** | ISR02_M04 |
| **미션 주제** | 생성형 AI 조작 이미지/텍스트 탐지 및 출처 계정 신뢰도 평가 모델 고도화 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 텔레그램·X 같은 SNS에 현장 제보와 함께 재게시, 수치·지역 변조, 과거 사진 재활용, 생성형 AI 조작물, 조직적 동시 유포가 한꺼번에 쏟아지면서 공개출처정보(OSINT)의 출처와 신뢰도를 빠르게 가려내는 일이 정보전의 핵심이 되었다. 본 프로젝트에서는 가상 계정과 가상 지역 코드로 만든 교육용 합성 게시물을 사용해 원천 데이터 스키마 정의, 텍스트 정제와 이미지 지각 해시 기반 중복 제거, 유사도 기반 복제·1차 출처 분류 Baseline, 조작·기만 탐지와 계정 신뢰도 결합 모델, 종합 신뢰도 등급(A~E) 엔진, 정보전 모니터링 대시보드, 정보관 긴급 검토(HITL), 대량 스트림 부하 시험, AAR 재보정, 최종 통합까지 단계별로 구현한다. 최종 목표는 조작 게시물을 놓치지 않으면서 사실 제보를 가짜로 몰지 않고, 파급력이 큰 첩보의 채택·차단은 원천 출처를 확인한 정보관이 결정하도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/isr02_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/` | 전처리, Baseline, 앙상블 탐지, 대시보드, HITL 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/isr02_data_spec.md`, `docs/isr02_hitl_guide.md`, `docs/isr02_aar_report.md`, `docs/isr02_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 군사 행정/지휘관 문서 |
| **`configs/`**| `configs/isr02_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/isr02_stress_test.py` | 500% 부하 시험 및 단위/통합 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/isr02_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 04. 생성형 AI 조작 이미지/텍스트 탐지 및 출처 계정 신뢰도 평가 모델 고도화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

생성형 AI 조작 텍스트/이미지 탐지 알고리즘과 작성자 계정 신뢰도 평가 모델을 결합하여 가짜 뉴스 판별 로직을 고도화한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **생성형 AI 조작 텍스트·이미지 탐지와 작성 계정 신뢰도 평가를 결합해 가짜 정보 판별 로직을 고도화한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **같은 이야기로 묶은 게시물 사이의 수량·지역·방향 불일치, 이미지 합성 잡음 불균일·생성형 격자 무늬·과거 이미지 재사용 흔적, 선동형 반복 문체, 조직적 동시 게시 특징 산출하고 모델에 적용하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/models/isr02_advanced_detector.py 및 예외 탐지 테스트 결과` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 기만/교란 공격에 대한 방어 로직의 현실성 및 신뢰성

#### 미션 PASS 조건

- 정량 기준: 복합 위협 탐지 재현율(Recall) 85% 이상, 처리 지연 200ms 이하


- 진행 기준: 복합 위협 시나리오 10종에 대한 탐지 테스트 통과


- 검증 방법: 합성 위협 데이터셋 주입 시 이상치 탐지 정확도 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""본문 내용 비교(수량·지역·방향 불일치), 이미지 조작 흔적(합성 잡음 불균일, 생성형 격자 무늬, 과거 이미지 재사용), 생성형·선동 문체, 조직적 동시 게시 특징과 작성 계정 신뢰도 모델을 결합해 10종의 조작·기만 게시물을 판별하는 모델로 고도화하고 재현율·오경보·처리 지연을 측정한다."""
import re
import time
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.fft import dctn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------------
# [교육용가상자료] 합성 SNS 게시물 생성기 (실제 계정·채널·지명·게시물 없음)
# 가상 사건마다 원본 제보, 재게시(복제), 유사 재작성, 조작·기만 게시물 10종과 32x32 합성 이미지를 만든다.
# ---------------------------------------------------------------------------
REGIONS = [f"{a}-{n}" for a in "ABCDEF" for n in range(1, 6)]
OBJECTS = ["군용 차량", "소형 드론", "보급 트럭", "헬기", "버스", "견인 차량", "트레일러", "민간 승용차"]
DIRS = ["북쪽", "남쪽", "동쪽", "서쪽"]
TEMPLATES = ["가상 {r} 지역에서 {o} 목격, {n}대 규모로 {d}으로 이동 중",
             "{r} 일대 {o} 확인됨. 약 {n}대, {d} 방향으로 이동",
             "[현지] {r}에서 {o} 발견 - 수량 {n}, 이동 방향 {d}",
             "{r} 주민 제보: {o} {n}대가 {d}으로 향하는 모습"]
MANIP = ["num_change", "region_swap", "dir_flip", "old_image", "splice", "ai_image", "ai_text", "bot_burst", "mashup", "recycled"]
ACCOUNT_TYPES = {  # 유형별 계정 속성 분포(가상): 나이(일), 팔로워, 하루 게시 수, 원본 비율, 과거 정확도, 게시 간격 변동계수
    "media": ((1000, 4000), (5e4, 5e5), (5, 30), (.6, .9), (.85, .97), (.8, 1.5)),
    "witness": ((10, 800), (20, 2000), (.2, 3), (.7, 1.0), (.6, .95), (1.0, 2.0)),
    "aggregator": ((200, 2000), (1e4, 2e5), (30, 200), (.05, .2), (.6, .85), (.5, 1.0)),
    "propaganda": ((5, 200), (100, 2e4), (20, 150), (.3, .7), (.1, .4), (.05, .3)),
}


def make_accounts(rng, n=(8, 30, 10, 20)):
    acc = {}
    for (t, spec), k in zip(ACCOUNT_TYPES.items(), n):
        for i in range(k):
            v = [rng.uniform(*r) for r in spec]
            acc[f"ACC_{t[:3].upper()}{i:02d}"] = {"type": t, "age_days": int(v[0]), "followers": int(v[1]), "posts_per_day": round(v[2], 1),
                                                   "original_ratio": round(v[3], 2), "past_accuracy": round(v[4], 2), "interval_cv": round(v[5], 2)}
    return acc


def event_image(seed):
    """사건 현장 사진을 흉내 낸 32x32 합성 이미지(사건마다 고유한 도형 배치)."""
    r = np.random.default_rng(seed)
    img = np.linspace(r.uniform(60, 120), r.uniform(120, 200), 32)[:, None] * np.ones((1, 32))
    for _ in range(int(r.integers(3, 7))):
        y, x, h, w = r.integers(0, 26, 4)
        img[y:y + 3 + h % 8, x:x + 3 + w % 8] = r.uniform(20, 240)
    return img + r.normal(0, 4, (32, 32))


def variant(img, rng):
    """재게시 과정의 자연스러운 변형: 재압축 잡음, 밝기 변화, 가장자리 잘림 중 하나."""
    k = rng.integers(0, 3)
    if k == 0:
        return ndimage.gaussian_filter(img, 0.6) + rng.normal(0, 3, img.shape)
    if k == 1:
        return img * rng.uniform(0.85, 1.15)
    return ndimage.zoom(img[2:30, 2:30], 32 / 28, order=1)


def splice(img, other, rng):
    out = img.copy()
    y, x = rng.integers(4, 18, 2)
    out[y:y + 12, x:x + 12] = other[y:y + 12, x:x + 12] + rng.normal(0, 12, (12, 12))   # 잡음 수준이 다른 조각을 붙임
    return out


def ai_image(rng):
    """생성형 모델 이미지를 흉내 낸 합성 이미지: 지나치게 매끈한 바탕 + 업샘플링 격자 무늬(주기 2픽셀)."""
    base = ndimage.gaussian_filter(rng.normal(130, 60, (32, 32)), 3)
    grid = (np.indices((32, 32)).sum(0) % 2) * rng.uniform(6, 12)
    return base + grid


def write(ev, rng, tpl=None, **kw):
    s = {"r": ev["r"], "o": ev["o"], "n": ev["n"], "d": ev["d"], **kw}
    return (tpl or TEMPLATES[int(rng.integers(len(TEMPLATES)))]).format(**s)


def make_posts(n_events=40, seed=7):
    rng = np.random.default_rng(seed)
    acc = make_accounts(rng)
    by = {t: [a for a, v in acc.items() if v["type"] == t] for t in ACCOUNT_TYPES}
    events = [{"id": f"EV{i:03d}", "r": str(rng.choice(REGIONS)), "o": str(rng.choice(OBJECTS)), "n": int(rng.integers(2, 15)),
               "d": str(rng.choice(DIRS)), "t0": float(i * 1.2 + rng.uniform(0, 1)), "img": event_image(seed * 1000 + i)} for i in range(n_events)]
    posts = []

    def add(ev, account, text, img, t, fwd, kind, origin):
        pid = f"P{len(posts):04d}"
        posts.append({"post_id": pid, "platform": str(rng.choice(["telegram", "x", "other"], p=[.6, .3, .1])), "account_id": account,
                      "posted_at": round(t, 3), "text": text, "image": img, "forwarded_from": fwd, "event_id": ev["id"] if ev else None,
                      "kind": kind, "is_disinfo": kind in MANIP, "origin": origin})
        return pid

    for k, ev in enumerate(events):
        src = str(rng.choice(by["witness"] if rng.random() < .6 else by["media"]))
        if rng.random() < .05:
            src = str(rng.choice(by["propaganda"]))                      # 신뢰도 낮은 계정도 사실을 올릴 수 있다
        o = add(ev, src, write(ev, rng), ev["img"], ev["t0"], None, "original", None)
        for _ in range(int(rng.integers(2, 5))):                          # 그대로 재게시(전달)
            add(ev, str(rng.choice(by["aggregator"])), posts[int(o[1:])]["text"], ev["img"], ev["t0"] + rng.exponential(1), o, "copy", o)
        for _ in range(int(rng.integers(1, 3))):                          # 표현만 바꾼 재작성(내용 동일)
            add(ev, str(rng.choice(by["media"] + by["witness"])), "#가상속보 " + write(ev, rng), variant(ev["img"], rng),
                ev["t0"] + rng.uniform(0.5, 6), None, "near_dup", o)
        for m in MANIP if k % 2 == 0 else MANIP[k % 4::4]:                # 조작·기만 게시물
            if rng.random() > 0.5:
                continue
            who = str(rng.choice(by["propaganda"] if rng.random() < .75 else by["aggregator"] + by["witness"]))
            t, img, text = ev["t0"] + rng.uniform(1, 12), variant(ev["img"], rng), None
            if m == "num_change":
                text = write(ev, rng, n=ev["n"] * int(rng.integers(5, 10)))
            elif m == "region_swap":
                text = write(ev, rng, r=str(rng.choice([r for r in REGIONS if r != ev["r"]])))
            elif m == "dir_flip":
                text = write(ev, rng, d={"북쪽": "남쪽", "남쪽": "북쪽", "동쪽": "서쪽", "서쪽": "동쪽"}[ev["d"]])
            elif m == "old_image" and k > 0:
                text, img = write(ev, rng), events[int(rng.integers(0, k))]["img"]
            elif m == "splice":
                text, img = write(ev, rng), splice(ev["img"], events[(k + 7) % n_events]["img"], rng)
            elif m == "ai_image":
                text, img = write(ev, rng), ai_image(rng)
            elif m == "ai_text":
                text = f"속보 속보! {ev['r']} 지역 대규모 {ev['o']} 출현! 긴급 확산 바람! 긴급 확산 바람! 모두 확산 바람!"
                img = ai_image(rng) if rng.random() < .5 else None
            elif m == "bot_burst":
                text = write(ev, rng, n=ev["n"] * 6) + " 지금 바로 공유"
                for b in rng.choice(by["propaganda"], 4, replace=False):
                    add(ev, str(b), text, img, t + rng.uniform(0, 0.15), None, m, o)
                continue
            elif m == "mashup":
                other = events[(k + 3) % n_events]
                text = f"{ev['r']} 일대 {ev['o']} 확인, {other['n']}대가 {other['d']}으로 이동"   # 다른 사건 내용과 짜깁기
            elif m == "recycled" and k > 2:
                old = events[int(rng.integers(0, k - 1))]
                text, img, t = write(old, rng), old["img"], ev["t0"] + rng.uniform(0, 3)
            if text is None:
                continue
            add(ev, who, text, img, t, None, m, o)
    return posts, acc
# ---------------------------------------------------------------------------

# ---- 텍스트 정제·이미지 지각 해시·조작 흔적 공통 함수 ----
STOPWORDS = {"가상", "약", "일대", "지역에서", "모습", "중", "확인됨", "발견", "제보", "주민"}
_URL = re.compile(r"https?://\S+|www\.\S+")
_SPECIAL = re.compile(r"[^0-9A-Za-z가-힣\-\s]")


def clean_text(t):
    """정규화(NFKC) -> URL·멘션 제거 -> 해시태그 기호 제거 -> 특수문자 제거 -> 불용어 제거 -> 공백 정리."""
    t = unicodedata.normalize("NFKC", t)
    t = _URL.sub(" ", t)
    t = re.sub(r"@\w+", " ", t).replace("#", " ")
    t = _SPECIAL.sub(" ", t)
    return " ".join(w for w in t.split() if w not in STOPWORDS)


def phash(img):
    """지각 해시(pHash): 32x32 이미지 DCT의 저주파 8x8(직류 성분 제외)을 중앙값과 비교한 64비트."""
    c = dctn(np.asarray(img, float), norm="ortho")[:8, :8].ravel()[1:]
    return np.append(c > np.median(c), False)


def hamming(a, b):
    return int(np.count_nonzero(a != b))


def slots(text):
    """본문에서 지역 코드·수량·방향·대상을 뽑는다(사건 내용 비교용)."""
    r = re.search(r"[A-F]-\d", text)
    n = re.search(r"(\d+)\s*대|수량\s*(\d+)", text)
    d = next((x for x in DIRS if x in text), None)
    o = next((x for x in OBJECTS if x in text), None)
    return {"r": r.group(0) if r else None, "n": int(n.group(1) or n.group(2)) if n else None, "d": d, "o": o}


def splice_score(img):
    """합성(붙여넣기) 흔적: 8x8 블록별 고주파 잡음 크기의 최대/중앙값 비율. 잡음 수준이 다른 조각이 있으면 커진다."""
    res = img - ndimage.median_filter(img, 3)
    b = np.abs(res).reshape(4, 8, 4, 8).std(axis=(1, 3))
    return float(b.max() / (np.median(b) + 1e-6))


def ai_image_score(img):
    """생성형 이미지 흔적: 2픽셀 주기 격자(나이퀴스트 주파수) 에너지 비율."""
    f = np.abs(np.fft.fft2(img - img.mean()))
    return float(f[16, 16] / (f.sum() / f.size + 1e-6))


def ai_text_score(text):
    """기계적 선동 문체 흔적: 반복 어절 비율 + 느낌표 밀도."""
    w = text.split()
    return (1 - len(set(w)) / max(len(w), 1)) + text.count("!") / max(len(w), 1)

# ---- 게시물 군집(같은 이야기 묶기)·조작 탐지 특징 공통 함수 ----
FEAT = ["num_dev", "region_mis", "dir_mis", "obj_mis", "img_ref_dist", "img_prior_h", "time_gap_h", "splice", "ai_img",
        "ai_text", "burst_n", "has_img", "is_fwd", "acc_age", "acc_followers", "acc_orig", "acc_past", "acc_cv"]


def cluster_posts(posts, hours=24, ham_thr=8):
    """같은 이야기 묶기: 본문에서 뽑은 (지역, 대상)이 같고 24시간 안이면 연결하고,
    지각 해시가 가까운 이미지(같은 대상)도 연결한다. 글 표현이 달라도 내용 단위로 묶인다."""
    texts = [clean_text(p["text"]) for p in posts]
    X = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)).fit_transform(texts)
    S = (X @ X.T).toarray()
    H = [phash(p["image"]) if p["image"] is not None else None for p in posts]
    sl = [slots(p["text"]) for p in posts]
    t = [p["posted_at"] for p in posts]
    parent = list(range(len(posts)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    for i in range(len(posts)):
        for j in range(i + 1, len(posts)):
            if abs(t[i] - t[j]) >= hours or sl[i]["o"] != sl[j]["o"]:
                continue
            same_story = sl[i]["r"] is not None and sl[i]["r"] == sl[j]["r"]
            same_img = H[i] is not None and H[j] is not None and hamming(H[i], H[j]) <= ham_thr
            if same_story or same_img:
                parent[find(i)] = find(j)
    return np.array([find(i) for i in range(len(posts))]), S, H


def post_features(posts, acc):
    cid, S, H = cluster_posts(posts)
    t = np.array([p["posted_at"] for p in posts])
    sl = [slots(p["text"]) for p in posts]
    first_seen = {}
    for i in np.argsort(t):                                           # 이미지가 처음 등장한 시각과 군집
        if H[i] is None:
            continue
        key = next((k for k in first_seen if hamming(np.array(k), H[i]) <= 6), None)
        if key is None:
            first_seen[tuple(H[i])] = (t[i], cid[i])
        elif first_seen[key][1] != cid[i]:
            pass
    ref = {c: int(np.argmin(np.where(cid == c, t, np.inf))) for c in set(cid)}   # 군집에서 가장 이른 게시물 = 기준(추정 원본)
    F = []
    for i, p in enumerate(posts):
        r = ref[cid[i]]
        a, b = sl[i], sl[r]
        img_prior = 0.0
        if H[i] is not None:
            for k, (ts, c) in first_seen.items():
                if hamming(np.array(k), H[i]) <= 6 and c != cid[i]:
                    img_prior = max(img_prior, min(t[i] - ts, 48.0))
        same_txt = [j for j in np.where(S[i] > 0.97)[0] if abs(t[j] - t[i]) < 0.25 and posts[j]["account_id"] != p["account_id"]]
        A = acc[p["account_id"]]
        F.append([abs(np.log((a["n"] or 1) / (b["n"] or 1))), float(a["r"] != b["r"]), float(a["d"] != b["d"]), float(a["o"] != b["o"]),
                  hamming(H[i], H[r]) if H[i] is not None and H[r] is not None else 0, img_prior, t[i] - t[r],
                  splice_score(p["image"]) if p["image"] is not None else 0, ai_image_score(p["image"]) if p["image"] is not None else 0,
                  ai_text_score(p["text"]), len(set(posts[j]["account_id"] for j in same_txt)), float(p["image"] is not None),
                  float(p["forwarded_from"] is not None), np.log1p(A["age_days"]), np.log1p(A["followers"]), A["original_ratio"],
                  A["past_accuracy"], A["interval_cv"]])
    return np.array(F), cid

CONTENT, ACCOUNT = list(range(13)), list(range(13, 18))

# 1) 학습: 가상 게시물(사건 60개) / 시험: 다른 난수의 가상 게시물(사건 60개)
tr_posts, tr_acc = make_posts(60, seed=7)
Ftr, _ = post_features(tr_posts, tr_acc)
ytr = np.array([p["is_disinfo"] for p in tr_posts])
te_posts, te_acc = make_posts(60, seed=11)
t0 = time.perf_counter()
Fte, cid = post_features(te_posts, te_acc)
yte, kind = np.array([p["is_disinfo"] for p in te_posts]), np.array([p["kind"] for p in te_posts])

# 계정 신뢰도 모델: 계정 속성만으로 '조작 게시물을 올릴 가능성'을 추정(작성자 평판)
acc_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Ftr[:, ACCOUNT], ytr)
Xtr = np.c_[Ftr[:, CONTENT], acc_model.predict_proba(Ftr[:, ACCOUNT])[:, 1]]
model = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=0).fit(Xtr, ytr)
Xte = np.c_[Fte[:, CONTENT], acc_model.predict_proba(Fte[:, ACCOUNT])[:, 1]]
pred = model.predict(Xte)
lat = (time.perf_counter() - t0) / len(te_posts) * 1000

content_only = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=0).fit(Ftr[:, CONTENT], ytr).predict(Fte[:, CONTENT])
account_only = acc_model.predict(Fte[:, ACCOUNT])
print("방식              조작 재현율  사실 게시물 오경보율")
for name, p in [("계정 신뢰도만", account_only), ("내용·이미지 탐지만", content_only), ("결합 모델", pred)]:
    print(f"{name:12s}  {(p & yte).sum() / yte.sum():10.1%}  {(p & ~yte).sum() / (~yte).sum():14.1%}")
rec = (pred & yte).sum() / yte.sum()
print(f"\n복합 위협(조작·기만) 탐지 재현율 {rec:.1%} (목표 85% 이상: {'달성' if rec >= 0.85 else '미달'}), 게시물당 처리 {lat:.2f} ms (목표 200 ms 이하)")

# 2) 조작·기만 시나리오 10종 시험(재현율 80% 이상이면 통과)
rows = [{"유형": m, "건수": int((kind == m).sum()), "재현율": round(float(pred[kind == m].mean()), 2),
         "판정": "통과" if pred[kind == m].mean() >= 0.8 else "미통과"} for m in MANIP]
t = pd.DataFrame(rows)
print("\n" + t.to_string(index=False))
print(f"통과 {int((t['판정'] == '통과').sum())}/10종")

# 3) 오경보 유발 패턴: 사실 게시물인데 조작으로 본 사례
fa = np.where(pred & ~yte)[0]
print(f"\n[사실 게시물 오경보 {len(fa)}건] 유형:", pd.Series(kind[fa]).value_counts().to_dict(),
      "/ 작성 계정 유형:", pd.Series([te_acc[te_posts[i]["account_id"]]["type"] for i in fa]).value_counts().to_dict())
imp = pd.Series(model.feature_importances_, index=[FEAT[i] for i in CONTENT] + ["account_risk"]).sort_values(ascending=False)
print("중요 특징 상위 5:", {k: round(v, 3) for k, v in imp.head(5).items()})
print("-> 신생·소규모 계정의 사실 제보와 늦게 올라온 재작성 글이 오경보를 만든다. 계정 평판에 과도하게 기대지 않도록 특징 공간을 재설계(미션 5)한다.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].barh(t["유형"], t["재현율"], color=["tab:green" if v >= 0.8 else "tab:red" for v in t["재현율"]]); ax[0].axvline(0.8, ls="--", c="k")
ax[0].set_title("Recall by manipulation type"); ax[0].set_xlim(0, 1.05)
imp.head(10)[::-1].plot.barh(ax=ax[1]); ax[1].set_title("Feature importance")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
방식              조작 재현율  사실 게시물 오경보율
계정 신뢰도만            88.3%            5.7%
내용·이미지 탐지만         94.1%           14.5%
결합 모델              97.3%            3.6%

복합 위협(조작·기만) 탐지 재현율 97.3% (목표 85% 이상: 달성), 게시물당 처리 3.14 ms (목표 200 ms 이하)

         유형  건수  재현율 판정
 num_change  13 1.00 통과
region_swap  24 0.88 통과
   dir_flip  19 0.95 통과
  old_image  29 1.00 통과
     splice  14 0.93 통과
   ai_image  16 1.00 통과
    ai_text  13 1.00 통과
  bot_burst  96 1.00 통과
     mashup  12 1.00 통과
   recycled  20 0.90 통과
통과 10/10종

[사실 게시물 오경보 12건] 유형: {'near_dup': 9, 'original': 3} / 작성 계정 유형: {'witness': 11, 'propaganda': 1}
중요 특징 상위 5: {'account_risk': 0.394, 'time_gap_h': 0.162, 'num_dev': 0.107, 'is_fwd': 0.102, 'img_ref_dist': 0.054}
-> 신생·소규모 계정의 사실 제보와 늦게 올라온 재작성 글이 오경보를 만든다. 계정 평판에 과도하게 기대지 않도록 특징 공간을 재설계(미션 5)한다.
```
</details>
